# Step 1: Mount Google Drive
This cell connects the cloud GPU to your personal Google Drive so it can access the uploaded dataset zip.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Cloning github repo

In [4]:
%cd /content
# Force delete the folder from Colab's hard drive
!rm -rf SSL_Prostate_Cancer_Grading

# Download a completely fresh, completely new copy straight from your branch
!git clone -b method/moco-v2 https://github.com/satvikkaul/SSL_Prostate_Cancer_Grading.git

# Move back in and list the training folder to prove it worked!
%cd SSL_Prostate_Cancer_Grading
!ls training


/content
Cloning into 'SSL_Prostate_Cancer_Grading'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 130 (delta 49), reused 105 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 9.44 MiB | 19.69 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/SSL_Prostate_Cancer_Grading
baseline  cae  moco  simclr


# Step 2: Extract Dataset to Cloud SSD
Reading images directly from Google Drive during training is extremely slow. 
This cell copies the zip file to the local lightning-fast Colab disk (`/content/`) and unzips it.

In [5]:
%cd /content/SSL_Prostate_Cancer_Grading

# Copy Zip from drive to this folder
!cp "/content/drive/MyDrive/Prostate_SSL/dataset.zip" ./

# Unzip it! (It will now perfectly create the dataset/ folder right next to your code)
!unzip -q dataset.zip


/content/SSL_Prostate_Cancer_Grading


# Step 3: Set up Requirements
Installs any libraries that aren't natively pre-installed on the Colab instance.

# Step 4: Run the Training Process!
Since the VS Code extension auto-syncs your local `.py` scripts, this command will execute your local script using the Colab GPU capabilities.
**CRITICAL Checkpoint Warning:** Update `CHECKPOINT_DIR` inside `pretrain_moco.py` to point to a Google Drive path (e.g., `'/content/drive/MyDrive/output/models'`) before you run this, otherwise Colab will delete your model weights when the session is closed! 

In [ ]:
!ls /content/SSL_Prostate_Cancer_Grading/training
!git checkout -b method/moco-v2


In [6]:
!git branch
!ls 
!nvidia-smi


* method/moco-v2
data	 dataset.zip  evaluation  README.md	    training
dataset  docs	      models	  requirements.txt  utils
Sun Mar  8 04:40:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   36C    P0             63W /  400W |   33226MiB /  81920MiB |      3%      Default |
|               

In [7]:

!python training/moco/pretrain_moco.py --epochs 20 --batch_size 128

2026-03-08 04:41:57.477833: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-08 04:41:57.544784: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
MoCo v2 Pre-training | RTX 3060 6GB | AMP=ON
  Epochs:      20
  Batch size:  128
  Queue K:     4096
  Momentum m:  0.999
  Temperature: 0.2
  Base LR:     0.03
Loaded manifest: 13831 images
Generator: 109 batches/epoch @ batch_size=128
2026-03-08 04:42:03.187551: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_F

In [8]:
# 1. Create a safe folder in your Google Drive to hold models
!mkdir -p /content/drive/MyDrive/Prostate_SSL/moco_models/

# 2. Copy the final encoder weights over from the Colab local disk
!cp ./output/models/moco/encoder_q_epoch020.weights.h5 /content/drive/MyDrive/Prostate_SSL/moco_models/

# 3. Optional: Copy the training log over too so you can chart the loss later!
!cp ./output/results/moco/logs/pretrain_log.csv /content/drive/MyDrive/Prostate_SSL/moco_models/


In [11]:
!git pull origin method/moco-v2


remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 444 bytes | 444.00 KiB/s, done.
From https://github.com/satvikkaul/SSL_Prostate_Cancer_Grading
 * branch            method/moco-v2 -> FETCH_HEAD
   77b3693..8e62f20  method/moco-v2 -> origin/method/moco-v2
Updating 77b3693..8e62f20
Fast-forward
 evaluation/moco/tsne_moco.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [12]:
# Point this to the exact .weights.h5 file you just saved to Drive!
!python evaluation/moco/tsne_moco.py --checkpoint /content/drive/MyDrive/Prostate_SSL/moco_models/encoder_q_epoch020.weights.h5 --n_samples 1000


2026-03-08 06:27:13.100596: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-08 06:27:13.167196: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading encoder from: /content/drive/MyDrive/Prostate_SSL/moco_models/encoder_q_epoch020.weights.h5
2026-03-08 06:27:18.439321: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1772951238.440320  102392 gpu_device.cc:2020] 

In [13]:
# Copy the PNG from the Colab project folder straight into your Google Drive Colab Models folder
!cp ./output/results/moco/tsne_pilot.png /content/drive/MyDrive/Prostate_SSL/moco_models/
